In [1]:
import pandas as pd
pd.options.display.float_format = '{:.2e}'.format
import numpy as np
import pickle as pkl
import matplotlib.pyplot as plt

from utils.utils import *

from InDelsTopo import Filtration, Complex

with open('data/preprocessed/Human_experiments_average.pickle', 'rb') as handle:
    Average_Experiments = pkl.load(handle)

In [2]:
import numpy as np
import random
from collections import Counter

def sample_geometric_tail(N):
    k = np.random.geometric(p=0.5) - 1
    return N + k

def random_word_poisson(n, alphabet=['a','b'],lam=1.0):
    """
    Generate a random word over {a,b} of length <= n,
    where the length is drawn from a Poisson(lam) distribution.
    """
    # Draw length from Poisson
    length = np.random.poisson(lam)
    
    # Cap at n
    length = min(length, n)
    
    # Generate random word
    return ''.join(random.choice(alphabet) for _ in range(length))

def random_word_geometric(n, alphabet=['a','b'], p=0.5):
    length = np.random.geometric(p) - 1   # shift to allow 0
    length = min(length, n)
    return ''.join(random.choice(alphabet) for _ in range(length))

def random_word_exponential(n, alphabet=['a','b'], scale=2.0):
    """
    Generate a random word over {a,b} of length <= n,
    where the length is drawn from an exponential distribution.
    
    scale = mean of the exponential distribution
    """
    length = int(np.random.exponential(scale=scale))
    length = min(length, n)
    
    return ''.join(random.choice(alphabet) for _ in range(length))

def random_word_weights(weights, alphabet=['a','b']):
    """
    Generate a random word over the given alphabet,
    where the word length is chosen randomly according to a list of weights.
    
    Parameters
    ----------
    weights : list of floats
        weights[i] is the relative probability of generating a word of length i.
        len(weights) = max length + 1
    alphabet : list of str
        symbols to use in the word
    
    Returns
    -------
    str
        Randomly generated word
    """
    # Ensure weights are a numpy array
    weights = np.array(weights, dtype=float)
    
    # Normalize weights to sum to 1
    weights /= weights.sum()
    
    # Choose the length according to weights
    length = np.random.choice(len(weights), p=weights)
    
    # Generate random word of that length
    return ''.join(random.choice(alphabet) for _ in range(length))

def random_word_weights_tail(weights,alphabet=['a','b']):
    """
    Generate a random word over the given alphabet,
    where the word length is chosen randomly according to a list of weights.
    If sum(weights)<1, with the remaining probability the length is chosen according to a
    geometric 2^x, with x>=N (the length of weights)
    
    Parameters
    ----------
    weights : list of floats
        weights[i] is the relative probability of generating a word of length i.
        len(weights) = max length + 1
    alphabet : list of str
        symbols to use in the word
    
    Returns
    -------
    str
        Randomly generated word
    """
    # Ensure weights are a numpy array
    weights = np.array(weights, dtype=float)
    p=sum(weights)

    if np.random.random()>=p:
        N=len(weights)
        length= sample_geometric_tail(N)
    else: 
        # Normalize weights to sum to 1
        weights /= weights.sum()
        # Choose the length according to weights
        length = np.random.choice(len(weights), p=weights)
    
    # Generate random word of that length
    return ''.join(random.choice(alphabet) for _ in range(length))

In [3]:
def random_word_table(N, alphabet, random_word_generator, **generator_kwargs):
    W=[]
    for _ in range(N):
        W.append(random_word_generator(alphabet=alphabet, **generator_kwargs))
    word_counts=Counter(W)

    words_table = pd.DataFrame({
    'word': list(word_counts.keys()),
    'count': list(word_counts.values())})

    words_table['weight']=words_table['count']/N

    return words_table

In [4]:
#Get weights according to the experiments
min_threshold=1e-5
max_N=12

labels=[]
all_weights=[]

for Experiment in Average_Experiments:
    labels.append(Experiment['exp_name'])
    Table=Experiment['table'].sort_values('avg_freq',ascending=True)

    #Filter by threshold for average frequency
    Table=Table[Table.avg_freq>=min_threshold]
    Table['length'] = Table['sequence'].str.len()
    Table['weight']=Table['avg_freq']/sum(Table['avg_freq'])
    Table=Table.groupby('length')['weight'].sum().reset_index()

    #Interpolate missing weights for i=0, ..., max_N
    total_weight=sum(Table[Table['length']<max_N]['weight'])
    weights2 = np.interp(range(max_N), Table['length'], Table['weight'])
    weights2=weights2/sum(weights2)*total_weight

    all_weights.append(weights2)

In [5]:
all_names=[]
all_reps=[]
all_dimensions=[]
all_betti0=[]
all_betti1=[]
all_betti2=[]
all_betti3=[]

N_random_words=250000
repetitions=3
min_count_treshold=2

for i in range(len(Average_Experiments)):
    print(i, end=': ')
    main_label=labels[i]
    weights=all_weights[i]
    for rep in range(repetitions):
        print(rep, end='')
        rep+=1
        random_table=random_word_table(N_random_words, ['A','T','C','G'], 
                               random_word_weights_tail, weights=weights)
        random_table_threshold=random_table[random_table['count']>=min_count_treshold]
        print('*', end='')

        K=Complex()
        K.compute_d_skeleton(random_table_threshold['word'],prod_symbol='', max_dim=4)

        print('*', end=', ')
        bettis=K.get_betti_numbers_z2(3)

        all_names.append(main_label)
        all_reps.append('rep_'+str(rep))
        all_dimensions.append(K.dim)
        all_betti0.append(bettis[0])
        all_betti1.append(bettis.get(1,0))
        all_betti2.append(bettis.get(2,0))
        all_betti3.append(bettis.get(3,0))
    print('.')

Big_Table= pd.DataFrame({
    "exp_name": all_names,
    'rep': all_reps,
    "dim": all_dimensions,
    "β₀": all_betti0,
    "β₁": all_betti1,
    "β₂": all_betti2,
    "β₃": all_betti3
})      

Big_Table = Big_Table.set_index(["exp_name","rep"])


0: 0**, 1**, 2**, .
1: 0**, 1**, 2**, .
2: 0**, 1**, 2**, .
3: 0**, 1**, 2**, .
4: 0**, 1**, 2**, .
5: 0**, 1**, 2**, .
6: 0**, 1**, 2**, .
7: 0**, 1**, 2**, .
8: 0**, 1**, 2**, .
9: 0**, 1**, 2**, .
10: 0**, 1**, 2**, .
11: 0**, 1**, 2**, .
12: 0**, 1**, 2**, .
13: 0**, 1**, 2**, .
14: 0**, 1**, 2**, .
15: 0**, 1**, 2**, .
16: 0**, 1**, 2**, .
17: 0**, 1**, 2**, .
18: 0**, 1**, 2**, .
19: 0**, 1**, 2**, .
20: 0**, 1**, 2**, .
21: 0**, 1**, 2**, .
22: 0**, 1**, 2**, .
23: 0**, 1**, 2**, .
24: 0**, 1**, 2**, .
25: 0**, 1**, 2**, .
26: 0**, 1**, 2**, .
27: 0**, 1**, 2**, .


In [6]:
with pd.option_context('display.max_rows', None):
    display(Big_Table.style \
    .format(lambda x: "." if x == 0 else x) \
    .apply_index(lambda vals: [f"color: {get_color_human(v)}" for v in vals], axis=0))

In [9]:
all_names=[]
all_reps=[]
all_dimensions=[]
all_betti0=[]
all_betti1=[]
all_betti2=[]
all_betti3=[]

N_random_words=250000
repetitions=3
min_count_treshold=2

for i in range(len(Average_Experiments)):
    print(i, end=': ')
    main_label=labels[i]
    weights=all_weights[i]
    for rep in range(repetitions):
        print(rep, end='')
        rep+=1
        random_table=random_word_table(N_random_words, ['A','T','C','G'], 
                               random_word_weights_tail, weights=weights)
        random_table_threshold=random_table[random_table['count']>=min_count_treshold]
        print('*', end='')

        K=Complex()
        K.compute_d_skeleton(random_table_threshold['word'],prod_symbol='', max_dim=4)

        print('*', end=', ')
        bettis=K.get_betti_numbers_z2(3)

        all_names.append(main_label)
        all_reps.append('rep_'+str(rep))
        all_dimensions.append(K.dim)
        all_betti0.append(bettis[0])
        all_betti1.append(bettis.get(1,0))
        all_betti2.append(bettis.get(2,0))
        all_betti3.append(bettis.get(3,0))
    print('.')

Big_Table= pd.DataFrame({
    "exp_name": all_names,
    'rep': all_reps,
    "dim": all_dimensions,
    "β₀": all_betti0,
    "β₁": all_betti1,
    "β₂": all_betti2,
    "β₃": all_betti3
})      

Big_Table = Big_Table.set_index(["exp_name","rep"])


0: 0**, 1**, 2**, .
1: 0**, 1**, 2**, .
2: 0**, 1**, 2**, .
3: 0**, 1**, 2**, .
4: 0**, 1**, 2**, .
5: 0**, 1**, 2**, .
6: 0**, 1**, 2**, .
7: 0**, 1**, 2**, .
8: 0**, 1**, 2**, .
9: 0**, 1**, 2**, .
10: 0**, 1**, 2**, .
11: 0**, 1**, 2**, .
12: 0**, 1**, 2**, .
13: 0**, 1**, 2**, .
14: 0**, 1**, 2**, .
15: 0**, 1**, 2**, .
16: 0**, 1**, 2**, .
17: 0**, 1**, 2**, .
18: 0**, 1**, 2**, .
19: 0**, 1**, 2**, .
20: 0**, 1**, 2**, .
21: 0**, 1**, 2**, .
22: 0**, 1**, 2**, .
23: 0**, 1**, 2**, .
24: 0**, 1**, 2**, .
25: 0**, 1**, 2**, 

KeyboardInterrupt: 

In [ ]:
with pd.option_context('display.max_rows', None):
    display(Big_Table.style \
    .format(lambda x: "." if x == 0 else x) \
    .apply_index(lambda vals: [f"color: {get_color_human(v)}" for v in vals], axis=0))

In [16]:
Parameters=[('Poisson', random_word_poisson,{'n':20}),
            ('Geometric', random_word_geometric,{'n':20}),
            ('Exponential', random_word_exponential,{'n':20}),
            ('Uniform',random_word_weights, {'weights':12*[1]})]

In [24]:
all_names=[]
all_reps=[]
all_dimensions=[]
all_betti0=[]
all_betti1=[]
all_betti2=[]
all_betti3=[]

N_random_words=100000
repetitions=3
min_count_treshold=5

for main_label,random_generator,params in Parameters:
    print(main_label, end=': ')
    for rep in range(repetitions):
        print(rep, end='')
        rep+=1
        random_table=random_word_table(N_random_words, ['A','T','C','G'], 
                               random_generator, **params)
        random_table_threshold=random_table[random_table['count']>=min_count_treshold]
        print('*', end='')

        K=Complex()
        K.compute_d_skeleton(random_table_threshold['word'],prod_symbol='', max_dim=4)

        print('*', end=', ')
        bettis=K.get_betti_numbers_z2(2)

        all_names.append(main_label)
        all_reps.append('rep_'+str(rep))
        all_dimensions.append(K.dim)
        all_betti0.append(bettis[0])
        all_betti1.append(bettis.get(1,0))
        all_betti2.append(bettis.get(2,0))
        #all_betti3.append(bettis.get(3,0))
    print('.')

Big_Table= pd.DataFrame({
    "exp_name": all_names,
    'rep': all_reps,
    "dim": all_dimensions,
    "β₀": all_betti0,
    "β₁": all_betti1,
    "β₂": all_betti2#,
    #"β₃": all_betti3
})      

Big_Table = Big_Table.set_index(["exp_name","rep"])


Poisson: 0**, 1**, 2**, .
Geometric: 0**, 1**, 2**, .
Exponential: 0**, 1**, 2**, .
Uniform: 0**, 1**, 2**, .


In [25]:
Big_Table= pd.DataFrame({
    "exp_name": all_names,
    'rep': all_reps,
    "dim": all_dimensions,
    "β₀": all_betti0,
    "β₁": all_betti1,
    "β₂": all_betti2#,
    #"β₃": all_betti3
})      

Big_Table = Big_Table.set_index(["exp_name","rep"])


In [27]:
with pd.option_context('display.max_rows', None):
    display(Big_Table.style \
    .format(lambda x: "." if x == 0 else x) \
    .apply_index(lambda vals: [f"color: {get_color_human(v)}" for v in vals], axis=0))